# VisionDrive: YOLO11n fine-tuning

Train a three-class bottle/cup/box detector using at least 60 unique training images per class. The published version 2 has 125/108/80 training images, 60 validation images, and 62 test images. Original COCO and ESPDet models remain separate.

This notebook launches the repository scripts. Downloads, training, and evaluation are opt-in; recorded results can be inspected without retraining. Use the project inference environment as the notebook kernel.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "scripts/train_yolo.py").is_file(), "Open this notebook from the project folder"
DATASET = ROOT / "datasets/yolo-bottle-cup-box-v2"
RUN = ROOT / "runs/yolo11n-bottle-cup-box"
RUN_RESTORE = False
RUN_TRAINING = False
RUN_EVALUATION = False

def run_script(name, *args):
    subprocess.run([sys.executable, str(ROOT / "scripts" / name), *map(str,args)], cwd=ROOT, check=True)


## 1. Restore the exact data

Source photos and existing annotations are from Open Images. Version 2 adds 33 box photos to training only; old validation/test splits are unchanged. File hashes and class image counts are audited. Do not split adjacent frames across partitions in future camera data.

Set `RUN_RESTORE=True` to allow public image downloads. Restore refuses to overwrite a dataset folder.

In [ ]:
if RUN_RESTORE:
    run_script("restore_dataset.py", "--manifest", ROOT / "data/public/yolo-bottle-cup-box-v2-manifest.jsonl", "--output", DATASET, "--download")

from collections import Counter
manifest = ROOT / "data/public/yolo-bottle-cup-box-v2-manifest.jsonl"
records = [json.loads(line) for line in manifest.read_text().splitlines()]
for split in ("train", "val", "test"):
    counts = Counter(label for row in records if row["split"] == split for label in {obj["label"] for obj in row["objects"]})
    print(split, dict(counts))


## 2. Fine-tune from pretrained YOLO11n

The CPU configuration uses 320-pixel training input, batch 8, 40 epochs maximum, early-stopping patience 12, AdamW with initial learning rate 0.001, seed 42, and a frozen 10-layer backbone. A three-class head replaces the COCO head. It does not retain the other 77 named categories.

Checkpoint selection uses native validation fitness. The test set is not used for model selection. Initial weights must exist at `weights/laptop/yolo11n.pt`; use the model setup instructions first. Training takes time and refuses an existing run folder.

In [ ]:
if RUN_TRAINING:
    run_script("train_yolo.py", "--dataset", DATASET, "--output", RUN)
else:
    print("Training disabled. Inspect the recorded results below or enable RUN_TRAINING for a new run.")


## 3. Evaluate the selected checkpoint

Confidence is selected by macro F1 on validation images only, then frozen for test scoring. Fixed-threshold metrics use class-aware one-to-one IoU >= 0.5 matching. AP uses the native validator and a separate confidence sweep. Evaluation runs at 640 pixels, matching the desktop viewer.

Do not adjust parameters based on test results and call the same set untouched. Use new independent camera sessions for whole-system evaluation.

In [ ]:
if RUN_EVALUATION:
    run_script("evaluate_finetuned_yolo.py", "--weights", RUN / "weights/best.pt", "--dataset", DATASET)
report_path = ROOT / "docs/yolo11-finetuned-evaluation.json"
if report_path.is_file():
    report = json.loads(report_path.read_text())
    print("Confidence:", report["configuration"]["confidence_threshold"])
    print(json.dumps(report["test"], indent=2))
    print(json.dumps(report["average_precision"], indent=2))
else:
    print("Recorded fine-tuning metrics have not been packaged yet.")


## 4. Run the trained model

From PowerShell in the project folder:

```powershell
.\run_logic_demo.cmd --weights weights/laptop/yolo11n-bottle-cup-box.pt --config config/yolo-finetuned.json
```

The default hybrid includes category-independent depth cues. The unchanged default COCO model remains available when no weights override is supplied. Detector metrics do not establish depth accuracy, instruction accuracy, or physical collision avoidance. See [the fine-tuning report](../docs/yolo11-finetuning-report.md) and [evaluation guide](../docs/evaluation-guide.md).